# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We'll demonstrate how to discover data entities by their `@id`, load data, process records, and visualize relationships.

### Dataset Source
The dataset source is defined by a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
name = dataset.metadata.name
description = dataset.metadata.description
print(f"{name}: {description}")

# Display the available record sets
if hasattr(dataset.metadata, 'record_sets'):
    print("Record sets found:")
    for rs in dataset.metadata.record_sets:
        print(f"- @id: {rs['@id']} | name: {rs['name']}")
else:
    print('No record sets found in metadata.')

## 2. Data Overview
Review available record sets, fields, and their IDs by listing them from the dataset.

In [ ]:
# List all record sets and their fields with their @id
if hasattr(dataset.metadata, 'record_sets'):
    for rs in dataset.metadata.record_sets:
        print(f"Record Set: {rs['name']} (ID: {rs['@id']})")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  Field: {field['name']} (ID: {field['@id']})")
        else:
            print("  No fields found in this record set.")
else:
    print("No record_sets attribute found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

In [ ]:
# Find all record set IDs
record_sets_ids = []
record_set_names = {}
if hasattr(dataset.metadata, 'record_sets'):
    for rs in dataset.metadata.record_sets:
        record_sets_ids.append(rs['@id'])
        record_set_names[rs['@id']] = rs['name']
else:
    print('No record sets found.')

dataframes = {}
# Load each record set into a pandas DataFrame referenced by its @id
for record_set_id in record_sets_ids:
    # Use mlcroissant's records method, referencing the record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_names[record_set_id]} (@id: {record_set_id})")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

# For demonstration, select the first record set
if record_sets_ids:
    selected_record_set_id = record_sets_ids[0]
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, e.g., filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# EDA example: Select numeric field and group field
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Using record set @id: {selected_record_set_id}")
    # Try to detect numeric fields
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Numeric field selected: {numeric_field_id}")
        # Arbitrary threshold for demonstration
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if available
        group_fields = [col for col in df.columns if df[col].dtype == 'object']
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical fields for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization for numeric field
if selected_record_set_id and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_record_set_id][numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id exists, visualize grouping
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[selected_record_set_id])
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` in accordance with FAIR principles. We reviewed available record sets, fields, and examined data distributions with simple visualizations. For further analysis, consult the Croissant schema and documentation for detailed field and column metadata.